Task 1. Generate n-grams from a corpus of text

In [1]:
import datasets
import transformers

In [2]:
#Load dataset
dset = datasets.load_dataset("imdb")

In [ ]:
#Make a tokenizer
base_tokenizer = transformers.AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokenizer = base_tokenizer.train_new_from_iterator(
    dset["train"]["text"],
    vocab_size=15000
)

In [4]:
# Now we tokenize the IMDB dataset the usual way
def tokenize(ex):
    return {"tokenized":tokenizer.tokenize(ex["text"])}

dset=dset.map(tokenize,num_proc=6)

Map (num_proc=6):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (588 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (762 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (704 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for thi

Map (num_proc=6):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (577 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (593 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (576 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (709 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (641 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for thi

Map (num_proc=6):   0%|          | 0/50000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1021 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (693 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1277 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (981 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for t

In [5]:
from collections import Counter
from more_itertools import sliding_window #more-itertools is an awesome library!
import tqdm

def generate_ngrams(dset,n):
    for ex in tqdm.tqdm(dset):
        tokens=["<bos>"]*(n-1)+ex["tokenized"]+["<eos>"] # <--- insert enough <bos> tokens to ensure we always have history of length n-1, and insert one <eos> token
        for ngram in sliding_window(tokens,n):
            yield ngram

Task 2. Count the n-grams

In [6]:
# Here we can concatenate all the individual datasets (train,test,unlabeled) in IMDB
# the "master" dataset is a dictionary of these, so dset.values() has the datasets of the individual sections (train,test,unlabeled)
combined_dataset=datasets.concatenate_datasets(list(dset.values()))

In [53]:
ngrams={} #This is the master dictionary
for ngram in generate_ngrams(combined_dataset,4): #let's start with 4-grams, you can try 3- and 5- grams too!
  if ngrams.get(f'{ngram[0]} {ngram[1]} {ngram[2]}') == None:
    ngrams[f'{ngram[0]} {ngram[1]} {ngram[2]}'] = {}
  if ngrams[f'{ngram[0]} {ngram[1]} {ngram[2]}'].get(ngram[3]) == None:
    ngrams[f'{ngram[0]} {ngram[1]} {ngram[2]}'][ngram[3]] = 0
  ngrams[f'{ngram[0]} {ngram[1]} {ngram[2]}'][ngram[3]] += 1

100%|██████████| 100000/100000 [01:29<00:00, 1123.51it/s]


Task 3. Generate new text

In [59]:
import numpy

def softmax(x):
    return numpy.exp(x)/sum(numpy.exp(x))

def sample_from(counts,temperature=1.0):
    """
    counts: list of counts that form the distribution
    temperature: the "how wild the generation should be" parameter, numbers close
                 to 0 are very conservative, numbers close or above 1 lead to quite
                wild generations
    """

    counts_array=numpy.array(counts)
    #Make these sum up to 1.
    counts_array_norm=counts_array/counts_array.sum()
    #Divide by temperature, that is what the algorithm does
    counts_array_norm/=temperature
    #Renormalize into a distribution using the softmax function, that is what the algorithm does
    final_distribution=softmax(counts_array_norm)
    #A good way to sample from a distribution is the following function from numpy
    x=numpy.random.multinomial(n=1,pvals=final_distribution)
    selected_word=numpy.argmax(x).flatten()
    return selected_word[0]

In [ ]:
from pprint import pprint

def generate(ngrams,n,max_len=40,temperature=1.0,prompt=None):
    """
    ngrams: the master dictionary
    n: the n in n-gram
    max_len: how many words max?
    temperature: the generation temperature
    prompt: the initial prompt, as a tuple, if not given n-1 <bos> symbols will be used
    """

    if prompt is None:
        prompt=["<bos>"]*(n-1) # <--- empty history means n-1 <bos> tokens

    generated=list(prompt) #this list will grow with words
    for _ in range(max_len):
        
        #Generate the key from the [i, i+1, i+2] words from the text
        key = f'{generated[_]} {generated[_+ 1]} {generated[_ + 2]}'
        #Get the value counts for this key
        counts = list(ngrams[key].values())
        #Using sample_from helper function pick the index of the next word
        index_of_next_word = sample_from(counts, temperature=temperature)
        #Using the index of the next key, choose that key as the next word
        next_word = list(ngrams[key].keys())[index_of_next_word]
        #Add the word to the list
        generated.append(next_word)

        if generated[-1]=="<eos>": #stop on end of sequence
            break
    return generated

# Now we can test it!

# make sure to match the n below to the n which was used to create
# the master dictionary
for temp in (0.01,0.1,0.5,1.0,2.0,5.0):
    generated=generate(ngrams=ngrams,n=4,max_len=60,temperature=temp)
    print(f"Temp={temp}:")
    pprint(" ".join(generated))
    print("-----------")

Temp=0.01:
("<bos> <bos> <bos> i ' m not sure if it was a foreign film . < br / > < br / "
 '> < br / > < br / > < br / > < br / > < br / > < br / > < br / > < br / > < '
 'br / > < br / >')
-----------
Temp=0.1:
('<bos> <bos> <bos> toxic zombies is a head - point on the perpet ##ra ##tor '
 "##s don ' t receive the same treatment to the genre by columbia or other b "
 'studios . credit to that . he wants to change paris ##he ##s , etc . bad '
 'movie . seriously , barney the purple dinosaur strikes as better '
 'entertainment . < br')
-----------
Temp=0.5:
('<bos> <bos> <bos> stinks ! " < br / > ` go ahead , but as opening credits '
 'started to roll ( unexpectedly ) prone to melodrama , but its 10 short '
 "stories on love somehow didn ' t counter the footage of scr ##im ##sh ##aw ( "
 "kevin mccarthy ) , kenneth mars ' s coach / principal is a tense uncertain")
-----------
Temp=1.0:
('<bos> <bos> <bos> greedy is a film project . as one , smooth , care ##f '
 '##ree gets across well 